# 情報システム概論　第6回

## 誤差逆伝播法

### どうして誤差逆伝播法について学ぶのか？

- 重みパラメータの勾配の計算を効率よく行うため

> 前回の数値微分による勾配の計算は時間がかかりましたよね．

### 計算グラフ

ここで計算グラフを使って，偏微分を計算することを考える．

まず，例として下記の式をxに関して偏微分することを考える．

$
z=(x+y)^2
$

この式は2つの式で構成される．

$
z=t^2\\
t=x+y
$

計算グラフで表すと下記のようになる．

<img alt="graph1.png" width="300" src="http://drive.google.com/uc?export=view&id=1uVTeRB1IPoS4AyslKIELWFx-EcWphHeW">

> xとyに対して足し算の処理をして，その結果をtに入れて，さらにtに対して2乗の処理をして，zに入れているんですね．  
このように処理を〇で表して，各処理の流れを→で表して，これらを組み合わせて処理の流れを計算グラフという図で表現しているんですね．

計算グラフを逆方向に進むと，偏微分を求めることができる ( ***逆伝播*** )．

> ちょっと直感的にわからないかもしれないので，どうやって偏微分を求めるのか以下で説明しますね．

<img alt="graph2.png" width="300" src="http://drive.google.com/uc?export=view&id=1WRHsRGSRpiFyKneT4ul4CAcplUFnG9j9">

> 仮に偏微分を求められるとして，上記のように書いてみましたという感じです．最終的な出力zを計算途中の変数であるt, x, yで偏微分するんですね．  
ちょっと思い出してください．前回は損失関数の値を各wで偏微分して勾配を計算していました．上記の図も同じことをしたいということです．まずは，ニューラルネットの学習のような複雑なものではなく，簡単な例で，最終的な出力であるzを途中の値で偏微分してみましょうということです．

> 前回は偏微分の計算は，実際にwをプラス方向にちょっと動かして損失関数を計算して，マイナス方向にちょっと動かして損失関数を計算して．．．ということをたくさんやって，数値微分の計算をしていたんですよね．これはすごく計算が面倒で，時間もかかります．なので，もっと簡単に計算できないか？ということで，何度も言いますが，まずは簡単な例でやってみましょう．

これは下記のように求められることがわかる．

<img alt="form_chain.png" width="300" src="http://drive.google.com/uc?export=view&id=1po7_0wJM1mO50hC-sxDRk5LA7W6Cn4xG">

> 計算グラフの一番後ろの∂z/∂zは1ということはわかると思います．後ろから2番目の∂z/∂tはz=t**2ですからtで偏微分したら2tになるのはわかりますよね．さらにその前の∂z/∂xは書き換えると∂z/∂t * ∂t/∂xになるのは合成関数の連鎖率です．一方で，∂z/∂tは2tで，∂t/∂xはt=x+yでしたから，xで偏微分したら1になりますよね．だから，∂z/∂x = 2t * 1になります．
> ここで何を言いたいかというと，一番後ろの偏微分は1，2番目の偏微分は2t (言い換えると1 * 2t)，3番目の偏微分は1 * 2t * 1 という風に，一番後ろから偏微分した結果をどんどん掛け算していけばよさそうだということです．

ここから，計算グラフの逆伝播を書き直すと以下のようになる．

<img alt="graph3.png" width="300" src="http://drive.google.com/uc?export=view&id=1e1jRdzHCQbayUL08SXZM1awbM-CvDkZ3">

> 上の図は後ろから順々にそれぞれの場所 (2乗であるとか，足し算であるとか) の偏微分の結果を掛け算していけばよいということで，うえで説明したことを図に表しているんですね．

一般的には以下のように表される．

<img alt="graph4.png" width="150" src="http://drive.google.com/uc?export=view&id=1pKwMkWfmVW0e43Ae1OA73fNGuFgaQEDz">

> fは上では2乗とか足し算のことで，fの後 (今はないけど右につながっているであろう) 処理での偏微分した結果をEとすると，処理fの入力であるxで偏微分した結果はE * ∂y/∂x になるよということです．これを数珠つなぎにすると，ニューラルネットのような複雑な計算の偏微分も簡単な計算で求められるというアイディアです．

ニューラルネットワークの勾配は，各重みに関する損失関数 (以下ではfだが，今後はLと表す事が多い)の偏微分のベクトルであり，

$
\left(\frac{\partial f}{\partial w_0},\frac{\partial f}{\partial w_1},\frac{\partial f}{\partial w_2},\cdots\right)
$

のように表された．これを逆伝播で求めるのが ***誤差逆伝播法*** である．

以降で，前に作ったニューラルネットワークの学習のプログラムを改造して，勾配を誤差逆伝播法で求める．
- 活性化関数にシグモイド関数を使っていたが，ReLU関数に変更する．
    - 多層になったとき(ディープラーニング)，シグモイド関数よりReLU関数のほうがうまく逆伝播できる．
- これまで作ったニューラルネットワークの各層 (レイヤ) に誤差逆伝播法の処理 (上で説明した手法で偏微分を計算する) を追加して，各層をクラスとして実装する．
    - 活性化関数
    - 重み付き信号の総和を計算する処理 (Affine)
    - 出力層の活性化関数(Softmax)＋損失関数
    
    > 活性化関数，出力層の活性化関数，損失関数は以前習ったのでわかると思います．出力層の活性化関数の計算と損失関数の計算を一緒にやると，誤差逆伝播の計算をしやすくなります．今は気にしないで，一緒に計算するんだね．．．と，深く考えずに納得いただければと思います．「重み付き信号の総和を計算する処理 (Affine)」と書いているますが，行列の積（重みの掛け算）を計算してからベクトルを足す（バイアスの足し算）という処理を一般にAffine変換というためです．
    
    > 上記の3つのクラスを作ってみましょう，ということです．
    
- こうすることで，数値微分で求めるより早く勾配を求められるはず
    - 数値微分では，ニューラルネットワークの順方向の計算を2回 (1つの重みを+hと-hずらして) 実施することを，重みの種類の回数だけ行う必要があった．
    - 誤差逆伝搬法では，ニューラルネットワークの逆方向に，1回ずつ，偏微分の掛け算をすればよい
    
    > 上でも少し言いましたが，前回やった数値微分はかなり時間がかかったと思います．誤差逆伝播法はニューラルネットの出力層から入力層に向けて，順々に偏微分の結果を掛け算していけば，勾配を求められそうなので，かなり計算量を減らせそうですね．ただ，ちょっとまだどうすればいいかイメージできないと思いますので，作りながら理解していきましょう．
    
    > これから作るのは下記の図のようなものでして，Affineのクラスと，ReLUのクラスと，Softmax+Lossのクラスです．それぞれのクラスには黒い四角で表しているforwardという関数があります．forward関数には今まで作ってきた処理が入ります．Affineのforwardは，前の層の出力と重みの行列を掛け算して，バイアスを足し算していた処理のことです．つまり，Affineのクラスが重みやバイアスを持っているということですね．ReLUのforwardはReLUの活性化関数の計算そのものです．Softmax+Lossのforwardも同じでSoftmaxの活性化関数の計算をしてから，交差エントロピー誤差で正解のラベルとの誤差を計算する処理そのものを表しています．これまでやってきたことをforwardという関数にしているということです．左から右に各クラスのforwardを数珠つなぎにすると，前回やった，例えば「5」の画像を入力してから，最終的に損失関数の値を計算するところまでできることがイメージできるでしょうか (下の例だと損失関数の計算結果として6.79...となっています)．ここまでは前回まででやっています．一方，backwardという関数も各クラスにあります．これは今回新たに作ろうとしているものです．上で説明したように，後ろから偏微分を計算して，1つ後ろの偏微分の計算結果と掛け算して．．．ということを数珠つなぎで計算していくための関数です．この計算をすると，最終的にはそれぞれのAffineクラスの中にある重み，バイアスの勾配 (∂L/∂Wなど) を計算できます．

<img alt="plan.png" width="600" src="http://drive.google.com/uc?export=view&id=1zZfBpEpbaxRmRiAfQRUrt9uA-KwJM0Li">

> なんとなく，全体像がつかめましたでしょうか．．．おそらく読んでいるだけだと，イメージしにくいと思いますので，実際に作ってみましょう．

### 誤差逆伝播ができる活性化関数レイヤ (ReLUレイヤ) を作る

ReLUは次の式で表された．

$
y = \begin{cases}
x & (x > 0) \\
0 & (x \leq 0)
\end{cases}
$

xに関するyの微分は以下のように求められる．

$
\frac{\partial y}{\partial x} = \begin{cases}
1 & (x > 0) \\
0 & (x \leq 0)
\end{cases}
$

- 順伝播 (入力から出力の方向のニューラルネットの計算) 時の入力xが0より大きいとき，逆伝播は値をそのまま流す．
> 1を掛け算しても変わりませんから，そのまま流すということですね．
- 順伝播時の入力xが0以下のとき，逆伝播では信号をストップ (0にする) する．

計算グラフで書くと以下のようになる．

<img alt="graph_relu.png" width="400" src="http://drive.google.com/uc?export=view&id=1d0lslUAbMk7CRwuGS0jrvdz_GCnSnn3y">

> Lは損失関数の値です．左はx>0の時で，この時は1つ後ろから来た前の偏微分の結果をそのまま(正確には1を掛け算しますが) 前に流します．右はx≦0の時で，この時は0を前に流します．

ReLUレイヤを作る．

In [1]:
import numpy as np

class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0

        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout

        return dx

ReLUクラスはインスタンス変数としてmaskを持つ．これはTrue/FalseからなるNumPy配列で，順伝播の入力であるxの要素で0以下の場所をTrue，それ以外をFalseとして保持する．少し例を使って説明する．

In [2]:
x = np.array([[1.0, -0.5], [-2.0, 3.0]])
x

array([[ 1. , -0.5],
       [-2. ,  3. ]])

In [3]:
mask = (x <= 0)
mask

array([[False,  True],
       [ True, False]])

> 上の例のように，xの行列の0以下のところだけTrueになっていることがわかりますね．  
forward関数の ```self.mask = (x <= 0)```でxのどこがマイナスになっているのかをmaskに保存します．その後，```out = x.copy()```で，入力であるxをoutにコピーしています．outとxは同じ行列になります．最後に```out[self.mask] = 0```をして，outのなかで，マイナスのところだけ0に置き換えています．この結果をforwardの出力にしています．ちょっと処理が複雑ですが，ReLUの処理をしていますよね．入力がマイナスだったら0になって，それ以外だったらそのままにするというのがReLUですからね．  
それで，このforwardの処理の中でmaskも得られました．これは入力のどこがマイナスかという情報が保存されています．backwardではこれを使って，∂L/∂xを計算します．```dout```は1つ後ろの偏微分の結果ですから，xがマイナスのところだけ```out[self.mask] = 0```で0に入れ替えているんですね．そして，この結果をReLUの偏微分の結果として，さらに1つ前に送ることになります．

Sigmoidレイヤは使わないので，今回は省略する．

### 誤差逆伝播ができるAffineレイヤを作る

Affineレイヤの順伝播における計算グラフは下記のようになる．
X, W, Bは行列である．例えば，下記の括弧の形状 (行の数, 列の数)で考える．なお，Nはバッチサイズである．

<img alt="graph_affine.png" width="300" src="http://drive.google.com/uc?export=view&id=1pCkbR2fWwF3Eharv_8OJ7ewt9mCi142m">

> とりあえずNは1として考えてみましょうか．2つの入力があって，これはXとしています．Xと2行3列の行列であるWとの内積をとっていますから，次の層のニューロンは3つなんですね．で，バイアスを足し算してその結果をYとしていますね．これは今までやってきたニューラルネットの計算ですよね．で，逆伝播はどうするかということですが，残念ながらちょっと複雑なので，説明は省かせていただきますが，以下のようになります．

X, W, Bに対する偏微分は下記のように求められる．

<img alt="graph_affine_back.png" width="400" src="http://drive.google.com/uc?export=view&id=1E2KuNyJ-Y5VRDwPYMyBTLOmP4ihYt6l4">

①，②，③を導く過程は省略する．③については少しわかりにくいと思うが，処理としては下記のようになる．

In [4]:
dY = np.array([[1,2,3],[4,5,6]])
dY

array([[1, 2, 3],
       [4, 5, 6]])

In [5]:
dB = np.sum(dY, axis=0)
dB

array([5, 7, 9])

> 1つ後ろの偏微分の結果は，∂L/∂Yですが，上記のプログラムではdYと表しています．それで，∂L/∂Bについては，∂L/∂Yの最初の軸(第0軸)に関する和ですが，これは上記例ですと上下で合計すればいいんですね．プログラムとしては，```np.sum(dY, axis=0)```で計算できます．

誤差逆伝播ができるAffineレイヤを下記のように実装する．

In [6]:
class Affine:
    def __init__(self, W, b):
        self.W =W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(self.x, self.W) + self.b

        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)

        return dx

> ```def __init__(self, W, b):```で重みやバイアスを初期化しています．
```
        self.W =W
        self.b = b   
```
の部分ですね．W, bは引数で指定されていて，Affineクラスを最初に使うときに設定できます．
```
        self.x = None
        self.dW = None
        self.db = None
```
の部分はxは入力，dW, dbはそれぞれ損失関数の値をWで偏微分した値と，bで偏微分した値で，これらが勾配になるんですね．
最初はなにも入っていませんのでNoneとして，空っぽにしています．  
```def forward(self, x):```はこれまでやってきた，入力xに対して重みWを掛け算して，バイアスbを足し算する処理をしていますね．  
```def backward(self, dout):```は新しく作ったところで，∂L/∂Wと∂L/∂bを計算しています．これらが勾配になります．
```
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
```
1つ目は∂L/∂xを計算しています．これは，上図の①の計算ですね．1つ後ろの偏微分の結果である∂L/∂YとWの転置の内積の計算をして，この関数での偏微分の結果として，1つ前に送ります．  
2つ目は∂L/∂Wを計算しています．これは，②の計算ですね．Xの転置と1つ後ろの偏微分の結果の内積の計算をしています．  
3つ目は∂L/∂Bを計算しています．これは，③の計算ですね．これは上で説明した通りですね．  
で，2つ目と3つ目は∂L/∂Wと∂L/∂Bを計算していて，これは損失関数の値を重み，バイアスで偏微分したものですから，勾配そのものになりますね．もともと勾配を簡単に計算することが目的でしたので，これで，どうやら勾配を得られそうです．


### 誤差逆伝播ができる Softmax-with-Loss レイヤを作る

ここでも導出過程を省くが，Softmax-with-Lossレイヤの計算グラフは下記のようになる (例として3クラス分類する場合)．

<img alt="graph_softmax_loss.png" width="400" src="http://drive.google.com/uc?export=view&id=1V-vHsUGGCweEbUVVKoczhjUYSbUiS73e">

ニューラルネットワークの学習は，ニューラルネットワークの出力と教師ラベルとの誤差を前のレイヤに伝えていくことで，勾配の計算を行う．ここでの(y1 - t1, y2 - t2, y3 - t3) はまさにSoftmaxレイヤの出力と教師ラベルとの誤差を素直に表している．

> ちょっと図が複雑ですが，一番左と右だけ見てください．a1～a2が出力層であるsoftmaxに入ると，右からLである損失関数の値が出てくるんですね．これは前回やってたsoftmaxの計算と損失関数の計算をしていることを1つの図にしているだけです．  
逆伝播だと，右から∂L/∂Lである1が入って，ここでの偏微分の結果であるy1-t1～y3-t3が出てきて，前の層に送ることを図に表しています．  
上記で書きましたように，このようにsoftmaxと交差エントロピー誤差の計算を一緒にすると偏微分の計算が単なる引き算になりますので，簡単になります．

具体的な例で考えると，例えばSoftmaxレイヤの出力が(0.3, 0.2, 0.5)で，教師ラベルが(0, 1, 0)だったとする．これは，正しく認識できていない．このときの逆伝播は(0.3, -0.8, 0.5)という大きな誤差を伝播することになる．

一方，Softmaxレイヤの出力が(0.01, 0.99, 0)の場合，逆伝播は(0.01, -0.01, 0)という小さな誤差になる．この小さな誤差が前レイヤに伝搬するが，その誤差は小さいため，前レイヤが学習する内容も小さくなる．

> たくさん間違っている場合は，重みやバイアスをたくさん変更しないといけないので，大きな値が偏微分の結果として，前の層に送られます．一方，正解している場合は，重みやバイアスをそれほど変更する必要がないので，小さな値が前の層に送られます．

誤差逆伝播ができる Softmax-with-Loss レイヤを実装するが，softmaxとcross_entropy_errorを使うので，以前作ったものを以下に持ってきて(コピペでOKです)，その後，Softmax-with-Lossを作る．

In [7]:
def softmax(x):
    if x.ndim == 2:
        x = x.T
        x = x - np.max(x, axis=0)
        y = np.exp(x) / np.sum(np.exp(x), axis=0)
        return y.T

    x = x - np.max(x) # オーバーフロー対策
    return np.exp(x) / np.sum(np.exp(x))

In [8]:
def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size

> 上の2つの関数は，前回作りましたので，コピペしていただいて結構です．

In [9]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None # softmaxの出力
        self.t = None # 教師データ

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)

        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size #データ1個あたりの誤差にする

        return dx

> 上はsoftmaxと交差エントロピー誤差の計算をいっぺんにするクラスです．
```def __init__(self):```の中で損失関数の値，出力であるy，ラベルであるtを空っぽの値Noneで初期化しています．  
```def forward(self, x, t):```ではこれまでやってきたsoftmaxの計算と交差エントロピー誤差の計算をやって，最終的な損失関数の計算結果を出力しています．  
```def backward(self, dout=1):```では，入力として，```dout=1```となっていますが，これは∂L/∂Lのことですね．そして，```dx = (self.y - self.t) / batch_size```のなかで，うえで説明したようにy-tの計算をしています．バッチでたくさんの画像をいっぺんに入力することを考えて，1つの画像当たりの誤差にするために，```batch_size```で割り算していますが，やっていることは上記で説明したことと変わりないことがわかります．



### 誤差逆伝播法に対応したニューラルネットワークの実装

これで誤差逆伝播法に対応したニューラルネットワークを作るための部品ができた．これらを使って，以前作った TwoLayerNet を改造する．下記の★印がついたところが変更点であるので，それ以外はコピペでOK．

In [10]:
from collections import OrderedDict #★

class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std = 0.01):
        # 重みの初期化
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        # レイヤの生成
        self.layers = OrderedDict() #★
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1']) #★
        self.layers['Relu1'] = Relu() #★
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2']) #★

        self.lastLayer = SoftmaxWithLoss() #★

    def predict(self, x): #★ 関数全体を書き換え
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    # x:入力データ, t:教師データ
    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1 : t = np.argmax(t, axis=1)

        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy

    def gradient(self, x, t): #★ 関数全体を書き換え
        # forward
        self.loss(x, t)

        # backward
        dout = 1
        dout = self.lastLayer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 設定
        grads = {}
        grads['W1'], grads['b1'] = self.layers['Affine1'].dW, self.layers['Affine1'].db
        grads['W2'], grads['b2'] = self.layers['Affine2'].dW, self.layers['Affine2'].db

        return grads

> 前回の``` __init__(self, input_size, hidden_size, output_size, weight_init_std = 0.01):```では重みとバイアスを初期化するだけでしたが，ここではニューラルネットの構成を追加しています．
```
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])
        self.lastLayer = SoftmaxWithLoss()
```
この部分ですね．```self.layers```の中にニューラルネットを構成する各層を入れようということで，最初に```self.layers = OrderedDict() ```とすることで，ディクショナリのような変数を作っています．これはリストのように順序どおりに値を持つこともできます．そして，この下の4行で各層の設定をしています．4行のうちの1行目で入力層から中間層への重みの掛け算とバイアスの足し算をするAffine1を追加して，その次の行で活性化関数としてReLUを設定しています．その下で，中間層から出力層への重みの掛け算とバイアスの足し算をするAffine2を追加して，最後にsoftmaxと損失関数の計算をするlastLayerを設定しています．出力層だけほかの層と違って，lastLayerという変数に入れています．  
```def predict(self, x):```は前回作ったときはここで重みの設定や，各層の処理(重みの掛け算，バイアスの足し算，活性化関数の計算)をやっていましたが，すでに```__init__```でそのための設定はself.layersという変数の中に入っていますので，これを使います．for文で1個1個取り出して，forward関数を呼び出すだけです．また，forward関数の出力は次のforward関数に入力することで，ニューラルネットの入力から出力への方向の計算を数珠つなぎでやっています．  
あと，前回と違うのは，```def gradient(self, x, t):```ですね．前回は```def numerical_gradient(self, x, t):```としていて，数値微分を計算していたんですが，今回は誤差逆伝播法で勾配を計算する方法ですので，新しい関数を作っています．まず，```self.loss(x, t) ```で入力から出力への計算をやり，ReLU関数のmaskの値や，Affineのxの値などを入れます (上で説明したように，maskなどは最初は空なので，一度はforward関数を実行して，値を入れてやる必要があります)．そして，
```
        dout = 1
        dout = self.lastLayer.backward(dout)
```
でsoftmaxと交差エントロピー誤差の偏微分を求めます．
```
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)
```
あとは，softmaxより前の層から，一番最初の層である入力層まで，後ろから前へ順々に偏微分の計算をしてあげればよいですね．上記の1～2行目で```self.layers```に入っている各層の順番を逆にしています．これは出力層から入力層への方向で，偏微分の計算をする必要があるからです．そして，for文で順番を逆にしたlayersから1個1個取り出して，順々に，偏微分の計算をしては，前の層の渡して，偏微分の計算をして．．．という風に繰り返します．  
これが終わると，Affine1，2のdWとdbの中には損失関数の値をWとbで偏微分した結果が入っているはずです．これは勾配で，これが欲しかったんですよね．勾配がわかると，重みとバイアスをどっち方向にずらせばよいかわかりますからね．  
最後に
```
        grads = {}
        grads['W1'], grads['b1'] = self.layers['Affine1'].dW, self.layers['Affine1'].db
        grads['W2'], grads['b2'] = self.layers['Affine2'].dW, self.layers['Affine2'].db
```
のようにして，勾配をgradsという変数に入れて，出力しています．


さらに，上で作ったニューラルネットワークを使って学習するためのプログラムも，以前作ったものを一部修正するだけでOK．

> 下記のプログラムのように，修正するところは
```
    grad = network.gradient(x_batch, t_batch)
```
ですね．以前は.numerical_gradientで数値微分していましたが，今回はgradientの関数を作りましたので，誤差逆伝播法で計算できるようになりました．  
では，準備ができましたので，実際に誤差逆伝播法で学習してみましょう．

In [11]:
from google.colab import drive
import sys

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/data'

Mounted at /content/drive


In [12]:
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
import os

with open(os.path.join(save_dir, 'mnist_X.pkl'), 'rb') as f:
  X = pickle.load(f)
with open(os.path.join(save_dir, 'mnist_y.pkl'), 'rb') as f:
  y = pickle.load(f)

x_train, x_test, t_train, t_test = train_test_split(X, y, test_size=10000, random_state=42)
t_train = np.eye(10)[t_train]   # 単位行列（対角成分が1の行列）を作成し，インデックスを指定
t_test = np.eye(10)[t_test]   # 単位行列（対角成分が1の行列）を作成し，インデックスを指定

In [13]:
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    # 勾配
    grad = network.gradient(x_batch, t_batch) #★

    # 更新
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)
    # print(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(train_acc, test_acc)

0.07893333333333333 0.0806
0.9007333333333334 0.8995
0.9251666666666667 0.9236
0.936 0.9333
0.9446666666666667 0.9405
0.9519833333333333 0.9457
0.9562166666666667 0.9491
0.9596166666666667 0.951
0.9639666666666666 0.9535
0.9661333333333333 0.9559
0.9693333333333334 0.9581
0.97075 0.9598
0.9728666666666667 0.9607
0.9738333333333333 0.9596
0.9757166666666667 0.9626
0.9771666666666666 0.9636
0.9776 0.9633


> どうでしょうか？一晩かかっても終わらなかった学習が，短時間で終わりましたよね．本当に学習できているんでしょうか？実際に試してみましょう．

In [17]:
np.argmax(network.predict(x_test[10]))

np.int64(3)

In [18]:
np.argmax(t_test[10])


np.int64(3)

In [19]:
network.accuracy(x_test, t_test)

np.float64(0.964)

> ちゃんと，3の画像が3であると言い当てていますね．また，精度については0.97くらいでほぼ正しく言い当てられていますね．  

> これで，ニューラルネットワークの学習ができるようになりました．ニューラルネットワークの作り方の基礎的なところはこれで全部学んだことになります．